# Grid world — exact dynamic programming

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alx87grd/minilink/blob/main/examples/learn/teaching/grid_world_exact_dp.ipynb)

This notebook implements the **exact dynamic programming algorithm** on a small heterogeneous grid world. Everything is plain `numpy` — no library to install — so that every line maps directly onto the algorithm as it is written on paper.

The point of the notebook is to see three things happen:

1. On a short horizon the **backward recursion** produces tables small enough to check by hand.
2. Increasing the horizon **changes the optimal policy**: with enough steps, the robot pays a geographic detour to avoid the mud.
3. Dynamic programming replaces an **exponential** search by a cost linear in the horizon.

The algorithm initializes with the terminal cost, then steps backward:

$$J^*_N(x) = g_N(x), \qquad J^*_k(x) = \min_{u \in \mathcal{U}(x)} \Big[\, g_k(x,u) + J^*_{k+1}\big(f_k(x,u)\big) \Big].$$

Here $x$ is the state, $u$ the action, $f_k$ the dynamics, $g_k$ the stage cost, $g_N$ the terminal cost, and $J^*_k(x)$ the **optimal cost-to-go**: the smallest total cost achievable from state $x$ with $N-k$ steps left.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

## 1. The problem

A robot moves on a 4x3 grid. Coordinates are `(i, j)` with `i` the column (1 to 4) and `j` the row (1 to 3).

- **Wall** at `(2,1)` and `(3,1)` — cannot be entered.
- **Mud** at `(2,2)` and `(3,2)` — passable, but five times more expensive.
- **Target** at `(4,1)` — free to sit on.

The state is the cell, the action is a unit move (or staying put), and the stage cost is a unit time penalty, quintupled in the mud:

$$g(x,u) = \begin{cases} 0 & x = x_\text{target} \\ 5 & x \in \mathcal{X}_\text{mud} \\ 1 & \text{otherwise} \end{cases}
\qquad
g_N(x) = \begin{cases} 0 & x = x_\text{target} \\ \infty & \text{otherwise} \end{cases}$$

The infinite terminal cost is what encodes *"reach the target within the horizon"*: any state from which the target is unreachable in the remaining steps inherits an infinite cost-to-go.

In [ ]:
WALL = {(2, 1), (3, 1)}
MUD = {(2, 2), (3, 2)}
TARGET = (4, 1)

# Admissible state set X: the full grid minus the wall
STATES = [(i, j) for i in (1, 2, 3, 4) for j in (1, 2, 3) if (i, j) not in WALL]

# Action set U: four cardinal moves plus staying put.
# "o" is listed first so that ties resolve to staying put (relevant only at the target).
ACTIONS = {"o": (0, 0), "^": (0, 1), "v": (0, -1), "<": (-1, 0), ">": (1, 0)}


def f(x, u):
    """Dynamics x_next = f(x, u). Bumping into the wall or the border leaves x unchanged."""
    di, dj = ACTIONS[u]
    x_next = (x[0] + di, x[1] + dj)
    return x_next if x_next in STATES else x


def g(x, u):
    """Stage cost g(x, u): a unit of time, quintupled in the mud, free at the target."""
    if x == TARGET:
        return 0.0
    return 5.0 if x in MUD else 1.0


def g_N(x):
    """Terminal cost: infinite unless the robot ended on the target."""
    return 0.0 if x == TARGET else np.inf

Before running anything, it helps to look at the world. The cell below draws it: the wall the robot cannot enter, the mud that costs five times as much to cross, the target, and the five actions available from any cell.

In [ ]:
def draw_grid(ax=None, title="The grid world"):
    """Draw the map: wall, mud, target, and the action set from cell (1,1)."""
    if ax is None:
        _, ax = plt.subplots(figsize=(5.2, 4.0))

    # Cells
    for i in (1, 2, 3, 4):
        for j in (1, 2, 3):
            if (i, j) in WALL:
                face, edge = "0.45", "0.15"
            elif (i, j) in MUD:
                face, edge = "0.80", "0.40"
            elif (i, j) == TARGET:
                face, edge = "0.94", "0.10"
            else:
                face, edge = "white", "0.65"
            ax.add_patch(plt.Rectangle((i - 0.5, j - 0.5), 1, 1,
                                       facecolor=face, edgecolor=edge, linewidth=1.6))

    ax.text(2.5, 1.0, "Wall", ha="center", va="center", color="white", fontweight="bold")
    ax.text(2.5, 2.0, "Mud\n(cost 5)", ha="center", va="center", fontsize=9)
    ax.text(*TARGET, "Target\n(4,1)", ha="center", va="center", fontsize=9, fontweight="bold")

    # Agent at (1,1) and the five actions available to it
    ax.add_patch(plt.Circle((1, 1), 0.22, facecolor="white", edgecolor="black", linewidth=1.6, zorder=3))
    ax.text(1, 1, "$x_k$", ha="center", va="center", fontsize=9, zorder=4)
    for dx, dy in ((0, 0.3), (0, -0.3), (-0.3, 0), (0.3, 0)):
        ax.arrow(1 + 0.26 * (dx > 0) - 0.26 * (dx < 0),
                 1 + 0.26 * (dy > 0) - 0.26 * (dy < 0),
                 dx, dy, head_width=0.08, color="black", zorder=3)

    ax.set_xlim(0.4, 4.6); ax.set_ylim(0.4, 3.6)
    ax.set_xticks([1, 2, 3, 4]); ax.set_yticks([1, 2, 3])
    ax.set_xlabel("$i$"); ax.set_ylabel("$j$")
    ax.set_aspect("equal")
    ax.set_title(title)
    return ax


draw_grid()
plt.tight_layout()
plt.show()

print("Admissible states:", len(STATES))
print("Actions:", list(ACTIONS))

## 2. The algorithm

The whole of exact dynamic programming is the loop below: one initialization, then one backward sweep per step, and inside it one minimization per state.

Note what the algorithm returns. Not a single trajectory, but a **time-varying closed-loop policy** — one action for every state, at every step — plus the optimal cost-to-go of every state.

In [ ]:
def exact_dp(N):
    """Backward recursion over N steps. Returns J[k][x] and pi[k][x] for k = 0..N."""
    J = [dict() for _ in range(N + 1)]
    pi = [dict() for _ in range(N + 1)]

    # 1. Initialization at the terminal step
    J[N] = {x: g_N(x) for x in STATES}
    pi[N] = {x: ("o" if x == TARGET else None) for x in STATES}

    # 2. Backward recursion, k = N-1 ... 0
    for k in range(N - 1, -1, -1):
        for x in STATES:
            Q = {u: g(x, u) + J[k + 1][f(x, u)] for u in ACTIONS}   # Q-values
            u_best = min(Q, key=Q.get)
            J[k][x] = Q[u_best]
            pi[k][x] = u_best if np.isfinite(Q[u_best]) else None   # undefined if unreachable

    return J, pi

## 3. Short horizon: checking the recursion by hand

With `N = 3` the tables are small enough to verify cell by cell with pen and paper. Read them bottom-up: row `j=1` is printed last.

`oo` marks an infinite cost-to-go — the target cannot be reached from that cell in the remaining number of steps.

In [ ]:
def show(table, kind="cost"):
    """Print a grid-shaped table, rows from j=3 down to j=1."""
    for j in (3, 2, 1):
        row = []
        for i in (1, 2, 3, 4):
            if (i, j) in WALL:
                row.append("###")
            elif kind == "cost":
                v = table[(i, j)]
                row.append(" oo" if not np.isfinite(v) else f"{v:3.0f}")
            else:
                row.append("  ." if table[(i, j)] is None else f"  {table[(i, j)]}")
        print("  ".join(row))


N = 3
J, pi = exact_dp(N)

for k in range(N, -1, -1):
    print(f"--- k = {k} " + ("(initialization)" if k == N else "") + " ---")
    show(J[k])
    print()

Two things are worth reading off these tables.

At `k = 0`, cell `(2,2)` has cost **11**: from there the only way to reach the target in three steps is straight through the mud (5 + 5 + 1). The short horizon leaves no room for a detour.

Cells `(1,1)`, `(1,2)`, `(1,3)` and `(2,3)` are still `oo`: they are simply too far. Their optimal action is undefined — any action is equally bad — which is why `exact_dp` returns `None` there.

## 4. Longer horizon: the detour appears

Now let the horizon grow. Watch the cost of starting at `(1,1)`, and watch the policy at `(2,2)`.

In [ ]:
print(" N   J*(1,1)   pi*(2,2)")
for N in range(1, 13):
    J, pi = exact_dp(N)
    cost = J[0][(1, 1)]
    print(f"{N:2d}   {'oo' if not np.isfinite(cost) else f'{cost:5.0f}':>7}   {str(pi[0][(2, 2)]):>8}")

The cost from `(1,1)` falls to **7** and then stops improving — that is the true optimal cost-to-go $J^*(1,1)$.

Note the policy at `(2,2)` flipping as the horizon grows. With few steps left the robot ploughs through the mud because it has no alternative; once the horizon is long enough, it prefers to climb out and go around. **The optimal action depends on how many steps remain**, which is exactly why a finite-horizon policy is generally non-stationary.

In [ ]:
J, pi = exact_dp(40)   # long enough to have converged

print("Optimal cost-to-go J*:")
show(J[0])
print("\nOptimal policy pi*:")
show(pi[0], kind="policy")

The detour costs `J*(1,1) = 7`. Compare that with the naive strategy of always heading straight at the target, which walks through both mud cells and costs 13 — you will reconstruct that number in the exercises below.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4.2))

field = np.full((3, 4), np.nan)
for (i, j), v in J[0].items():
    field[3 - j, i - 1] = v

im = ax.imshow(field, cmap="viridis_r", extent=(0.5, 4.5, 0.5, 3.5))
fig.colorbar(im, ax=ax, label="$J^*(x)$")

arrow = {"^": (0, 0.3), "v": (0, -0.3), "<": (-0.3, 0), ">": (0.3, 0)}
for (i, j), u in pi[0].items():
    if u in arrow:
        dx, dy = arrow[u]
        ax.arrow(i - dx / 2, j - dy / 2, dx, dy, head_width=0.09, color="white")
    elif u == "o":
        ax.plot(i, j, "w*", markersize=14)

for (i, j) in WALL:
    ax.add_patch(plt.Rectangle((i - 0.5, j - 0.5), 1, 1, color="black"))
for (i, j) in MUD:
    ax.add_patch(plt.Rectangle((i - 0.5, j - 0.5), 1, 1, fill=False,
                               edgecolor="orangered", linewidth=3))

ax.set_xticks([1, 2, 3, 4]); ax.set_yticks([1, 2, 3])
ax.set_title("Optimal cost-to-go and policy (mud outlined in red)")
plt.tight_layout()
plt.show()

## 5. Why not brute force?

Searching for the optimal policy by enumeration is hopeless, and the numbers are worth producing yourself.

With $S$ states, $M$ actions and a horizon of $N$ steps:

| Quantity | Meaning |
| --- | --- |
| $M^N$ | trajectories from **one** starting state |
| $S \times M^N$ | naive evaluation of the trajectory tree from every starting state |
| $M^{S \times N}$ | size of the space of **all** policies |
| $N \times S \times M$ | cost of **dynamic programming** |

The last line is the whole point: by storing the cost-to-go, the recursion flattens the exponential tree into something linear in the horizon.

In [ ]:
S, M = len(STATES), len(ACTIONS)

print(f"S = {S} states, M = {M} actions\n")
for N in (3, 15, 30):
    print(f"horizon N = {N}")
    print(f"  all policies      M^(S*N) = {M}^{S * N} ~ 1e{S * N * np.log10(M):.0f}")
    print(f"  naive tree        S*M^N   ~ 1e{np.log10(S) + N * np.log10(M):.0f}")
    print(f"  dynamic prog.     N*S*M   = {N * S * M}")
    print()

For a horizon of 15 steps on this tiny 10-state grid, the space of policies already exceeds the estimated number of atoms in the observable universe ($\approx 10^{80}$), while dynamic programming settles the same problem in a few hundred operations.

The catch is the state count $S$. Here $S = 10$. Discretize four continuous variables into 100 values each and $S = 100^4 = 10^8$ — the recursion is still linear in the horizon, but each sweep has become unaffordable. That is the **curse of dimensionality**, and it is what motivates approximate methods: discretization, function approximation, sampling, and online search.

## 6. Things to try

1. **Move the mud.** Put mud on the top row instead and check that the optimal policy switches back to the direct route.
2. **Reprice the mud.** Lower the mud cost from 5 to 2. At what price does the detour stop being worth it? Predict the answer before running it.
3. **Remove the wall.** How does $J^*(1,1)$ change, and why by that much?
4. **Break the terminal cost.** Replace `np.inf` by a large finite value (say 1000) in `g_N`. The policy is unchanged, but the cost-to-go of unreachable states is no longer infinite — explain what those numbers now mean.
5. **Evaluate a fixed policy.** Write the naive "always head at the target" policy as a dictionary, simulate it forward from `(1,1)`, and check that its cost is 13. Note that evaluating one policy needs no minimization at all — that is the difference between *evaluation* and *optimization*.